# Lezione 08 — Gestione degli Errori: Eccezioni

## Obiettivi della lezione

Al termine di questa lezione sarai in grado di:

- Spiegare la differenza tra un **errore** (grave, dell'ambiente/sistema) e un'**eccezione** (recuperabile, generata dal tuo stesso codice in esecuzione).
- Usare `try`/`except` per evitare che il tuo programma si fermi quando qualcosa va storto.
- Catturare **eccezioni specifiche** (`ZeroDivisionError`, `ValueError`, `IndexError`, `KeyError`) invece di usare un `except` generico.
- Combinare più clausole `except`, raggruppare eccezioni in una tupla, e usare un `except` generico come rete di sicurezza finale.
- Usare `else` (codice che gira solo se non ci sono state eccezioni) e `finally` (codice che gira sempre).
- Scrivere funzioni robuste che gestiscono errori di input senza far crashare il programma.


## Perché ti dovrebbe interessare?

Finora hai scritto codice assumendo che tutto andasse bene. Nella vita reale della data science questo quasi non succede mai: file che non sono dove te li aspettavi, colonne di testo con un `"N/A"` infilato in mezzo a numeri, dizionari a cui manca una chiave perché il dataset ha cambiato formato.

Senza gestione delle eccezioni, uno qualsiasi di questi problemi manda a picco l'intero programma. Con la gestione delle eccezioni, decidi tu: salti quella riga, avvisi l'utente, usi un valore predefinito — ma il programma resta vivo.


## 1. Errori vs. eccezioni: non sono la stessa cosa

Un **errore** è un problema serio che deriva dall'ambiente, dall'hardware o da errori gravi di sintassi. Di solito è così severo che il programma non può continuare a funzionare; normalmente non lo "gestisci" a runtime, lo correggi prima di eseguire il codice (ad esempio, un `SyntaxError`).

Un'**eccezione** nasce dall'esecuzione del tuo codice in una situazione concreta: hai diviso per zero, hai convertito un testo non numerico, hai chiesto una posizione che non esiste. **Puoi catturarle e gestirle** con `try`/`except`, lasciando che il programma continui.

| Aspetto | Errori | Eccezioni |
|---|---|---|
| Origine | Ambiente, hardware, sistema | Esecuzione problematica del tuo codice |
| Gravità | Gravi, bloccano il programma | Meno gravi, si possono gestire |
| Gestione | Non si catturano con `try`/`except` | Si catturano con `try`/`except` |
| Esempi | `SyntaxError`, errori di configurazione | `ZeroDivisionError`, `ValueError`, `KeyError` |


In [ ]:
# Un'eccezione: Python può sollevarla a runtime,
# ma il codice in sé è sintatticamente corretto.
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("È stata lanciata un'eccezione:", type(e).__name__, "->", e)

# Un errore di sintassi non arriva nemmeno a essere eseguito; Python si
# lamenterebbe prima di eseguire il programma, e per questo non si "cattura"
# mai con try/except -- si corregge direttamente nel codice. Ad esempio
# (non viene eseguito, solo per illustrare):
#     def calcolare(x
#         return x * 2
#     SyntaxError: '(' was never closed


## 2. `try`/`except`: la rete di sicurezza di base

Il codice che può fallire va dentro il blocco `try`. Se non succede nulla di strano, Python lo esegue per intero e prosegue, ignorando l'`except`. Se si verifica un'eccezione, Python interrompe il `try` esattamente dove ha fallito e salta direttamente all'`except` corrispondente. Dopo, il programma continua normalmente — non si ferma.


In [ ]:
a = 1
b = 0  # cambia questo valore (ad esempio a 5) e riesegui la cella

try:
    resultado = a / b
    print("Successo, risultato =", resultado)
except ZeroDivisionError:
    print("Errore: non si può dividere per zero")

# Questa riga viene eseguita in ogni caso, a prescindere da cosa succede nel try/except
print("Il programma continua a funzionare normalmente")


Puoi scrivere un `except` senza specificare nessun tipo di eccezione (un **except generico**), e catturerà qualsiasi cosa vada storta dentro il `try`. Funziona, ma come vedrai subito, nasconde informazioni preziose.


In [ ]:
valores_a_probar = ["5", "abc"]

for texto in valores_a_probar:
    try:
        numero = int(texto)
        resultado = 10 / numero
        print("Risultato:", resultado)
    except:
        print(f"Qualcosa è andato storto elaborando {texto!r}")


## 3. Eccezioni specifiche: perché l'`except` generico ti nasconde i problemi

Il problema di `except:` da solo è duplice: (1) nasconde informazioni — se il tuo codice fallisce per un bug reale che non avevi previsto, l'`except` generico lo inghiotte in silenzio come se fosse un'eccezione "normale", e ti ritrovi senza indizi; (2) ti impedisce di reagire in modo diverso a seconda del problema — non è la stessa cosa che l'utente abbia scritto del testo (`ValueError`) o che abbia diviso per zero (`ZeroDivisionError`).

La soluzione è catturare tipi specifici di eccezione, ciascuno con il proprio blocco `except`:


In [ ]:
def dividir(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
        print(f"Successo, risultato = {resultado}")
    except ZeroDivisionError:
        print("Il numero che hai dato non può dividere perché è zero")
    except ValueError:
        print("Non hai inserito un numero valido")

dividir(10, "2")
dividir(10, "0")
dividir(10, "abc")


Puoi anche raggruppare più tipi in un'unica clausola se vuoi dare loro lo stesso trattamento, usando una tupla:


In [ ]:
try:
    resultado = 1 / 0
except (ZeroDivisionError, NameError):
    # si esegue se si verifica uno qualsiasi di questi due tipi
    print("Si è verificato un problema numerico o di variabile")


Una buona pratica è combinare clausole specifiche con un `except` generico **alla fine**, come rete di sicurezza per ciò che non avevi previsto — ma mai come unica linea di difesa. Python valuta gli `except` in ordine, dall'alto verso il basso, ed esegue il primo che corrisponde al tipo di eccezione lanciata.


In [ ]:
def dividir_seguro(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
        print(f"Successo, risultato = {resultado}")
    except ZeroDivisionError:
        print("Il numero che hai dato non può dividere perché è zero")
    except ValueError:
        print("Non hai inserito un numero valido")
    except Exception:
        print("È successo qualcosa di inaspettato")

dividir_seguro(10, "5")
dividir_seguro(10, "0")
dividir_seguro(10, "dieci")
dividir_seguro(10, None)   # int(None) è un TypeError -> finisce nell'except generico


## 4. `else` e `finally`: codice che dipende (o no) dal risultato

`else` esegue codice solo quando il blocco `try` è terminato **senza nessuna eccezione** — utile per separare "ciò che può fallire" da "ciò che voglio fare solo se non è fallito nulla". `finally` va un passo oltre: si esegue **sempre**, ci sia stata o no un'eccezione, e si usa tipicamente per operazioni di pulizia (chiudere un file, segnalare che l'elaborazione è terminata).

L'ordine è sempre: `try`, uno o più `except`, opzionalmente `else`, e opzionalmente `finally` alla fine.


In [ ]:
def dividir_completo(a, texto_b):
    try:
        b = int(texto_b)
        resultado = a / b
    except ZeroDivisionError:
        print("Il numero che hai dato non può dividere perché è zero")
    except ValueError:
        print("Non hai inserito un numero valido")
    else:
        # Si esegue solo se non ci sono state eccezioni
        print("Successo, risultato =", resultado)
    finally:
        # Si esegue sempre, ci sia stata o no un'eccezione
        print("Elaborazione completata")
    print()  # riga vuota per separare ogni prova

dividir_completo(10, "5")
dividir_completo(10, "0")
dividir_completo(10, "dieci")


## 5. Tour tra le eccezioni più comuni lavorando con i dati

Hai già visto `ZeroDivisionError` (dividere per zero) e `ValueError` (valore inappropriato, come convertire testo non numerico). Le altre due che incontrerai continuamente lavorando con liste e dizionari sono:

- **`IndexError`** — hai chiesto una posizione di una lista fuori dal suo intervallo valido.
- **`KeyError`** — hai chiesto una chiave di un dizionario che non esiste. Nota che `.get()` evita questo errore restituendo `None` (o un valore predefinito) invece di lanciare l'eccezione.

(Ne esistono molte altre: `FileNotFoundError`, `TypeError`, `AttributeError`, `ImportError`, tra le altre — riconoscere il nome in un traceback ti dice subito cosa è andato storto.)


In [ ]:
inventario = ["mela", "pera", "uva"]
notas = {"anna": 9, "beto": 7}

try:
    print(inventario[5])
except IndexError as e:
    print("IndexError:", e)

try:
    print(notas["carla"])
except KeyError as e:
    print("KeyError:", e)

# .get() evita il KeyError restituendo un valore predefinito
print("Con .get():", notas.get("carla", "nessuna nota registrata"))


## 6. Esercizi risolti: la stessa logica, come funzioni riutilizzabili

### Esercizio 1 — `safe_divide`

Una funzione che divide due numeri e gestisce `ZeroDivisionError` restituendo `None` invece di far crashare il programma.


In [ ]:
def safe_divide(numerator, denominator):
    try:
        result = numerator / denominator
        return result
    except ZeroDivisionError:
        print("Errore: impossibile dividere per zero.")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(7, -2))


### Esercizio 2 — `calcular_raiz`

Una funzione che calcola la radice quadrata con `math.sqrt()` e distingue **due** motivi diversi di `ValueError`: un testo non numerico (fallisce la conversione con `float()`) e un numero negativo (fallisce `math.sqrt()`). Usa un `try` annidato dentro l'`except` per distinguerli, e un `finally` che avvisa sempre che ha terminato.


In [ ]:
import math

def calcular_raiz(entrada):
    try:
        numero = float(entrada)
        resultado = math.sqrt(numero)
        print(f"La radice quadrata di {numero} è {resultado}")
    except ValueError:
        try:
            float(entrada)
            # se la conversione funziona, il problema era il numero negativo
            print("Input non valido: inserisci un numero positivo")
        except ValueError:
            # se anche la conversione fallisce, il testo non era numerico
            print("Input non valido: non hai inserito un numero")
    finally:
        print("Calcolo terminato")

for entrada in ["16", "-4", "dieci"]:
    calcular_raiz(entrada)
    print()


### Esercizio 3 — `complex_calculation`

Una funzione che calcola `num / (num - 5)` e cattura **qualsiasi** eccezione con `except Exception as e`, indipendentemente dal tipo.


In [ ]:
def complex_calculation(num):
    try:
        result = num / (num - 5)
        print(f"Risultato: {result}")
    except Exception as e:
        print("Si è verificato un errore durante il calcolo.")

complex_calculation(10)   # funziona senza problemi
complex_calculation(5)    # num - 5 == 0 -> ZeroDivisionError
complex_calculation("abc")  # "abc" - 5 -> TypeError; l'except generico lo cattura comunque


## Riepilogo: cosa porti a casa da questa lezione

- Un **errore** è grave e deriva dall'ambiente (`SyntaxError`); non si gestisce con `try`/`except`. Un'**eccezione** nasce dal tuo codice in esecuzione e si può catturare e gestire.
- `try` contiene il codice che può fallire; se si verifica un'eccezione, Python salta all'`except` corrispondente e il programma continua.
- Catturare tipi specifici (`except ZeroDivisionError:`, `except ValueError:`) è una pratica migliore di un `except:` generico: ti permette di reagire diversamente a ogni problema e non nasconde bug che non avevi previsto.
- Puoi raggruppare più tipi in una tupla `except (TipoA, TipoB):`, e combinare clausole specifiche con una generica **alla fine** come rete di sicurezza.
- `else` gira solo se il `try` è terminato senza errori; `finally` gira sempre, ideale per operazioni di pulizia.
- Le eccezioni più comuni lavorando con i dati: `ZeroDivisionError`, `ValueError`, `IndexError`, `KeyError` (e anche `FileNotFoundError`, `TypeError`, `AttributeError`, `ImportError`).
- `.get()` nei dizionari evita un `KeyError` senza bisogno di `try`/`except`.
